In [2]:
import pandas as pd
import sqlite3

In [3]:
df = pd.read_excel('target_financials_raw.xlsx')

In [5]:
conn = sqlite3.connect('target_analysis.db')

In [6]:
df.to_sql('financials_raw',conn,if_exists='replace',index=False)
print(pd.read_sql("SELECT * FROM financials_raw",conn))

  fiscal_year        period_ending  net_sales  cost_of_sales  gross_profit  \
0      FY2021  2021-01-30 00:00:00      93561          66177         27384   
1      FY2022  2022-01-29 00:00:00     106005          74963         31042   
2      FY2023  2023-01-28 00:00:00     109120          82306         26814   
3      FY2024  2024-02-03 00:00:00     107412          77828         29584   
4      FY2025  2025-02-01 00:00:00     106566          76502         30064   

   sga_expenses  operating_income  net_income  
0         18546              6608        4368  
1         19636              9062        6946  
2         20469              3960        2780  
3         21269              5900        4138  
4         21796              5739        4091  


In [ ]:
query = """
CREATE TABLE financials_computed AS
SELECT
    fiscal_year,
    period_ending,
    net_sales,
    cost_of_sales,
    gross_profit,
    sga_expenses,
    operating_income,
    net_income,

    -- Margin calculations 
    ROUND(gross_profit * 100.0 / net_sales, 2) AS gross_margin_pct,
    ROUND(sga_expenses * 100.0 / net_sales, 2) AS sga_pct,
    ROUND(operating_income * 100.0 / net_sales, 2) AS operating_margin_pct,

    -- Cost of sales as % of revenue (inverse of gross margin)
    ROUND(cost_of_sales * 100.0 / net_sales, 2) AS cogs_pct

FROM financials_raw
ORDER BY period_ending;
"""

conn.execute("DROP TABLE IF EXISTS financials_computed")
conn.execute(query)
conn.commit()

result = pd.read_sql("SELECT * FROM financials_computed",conn)
print(result)

  fiscal_year        period_ending  net_sales  cost_of_sales  gross_profit  \
0      FY2021  2021-01-30 00:00:00      93561          66177         27384   
1      FY2022  2022-01-29 00:00:00     106005          74963         31042   
2      FY2023  2023-01-28 00:00:00     109120          82306         26814   
3      FY2024  2024-02-03 00:00:00     107412          77828         29584   
4      FY2025  2025-02-01 00:00:00     106566          76502         30064   

   sga_expenses  operating_income  net_income  gross_margin_pct  sga_pct  \
0         18546              6608        4368             29.27    19.82   
1         19636              9062        6946             29.28    18.52   
2         20469              3960        2780             24.57    18.76   
3         21269              5900        4138             27.54    19.80   
4         21796              5739        4091             28.21    20.45   

   operating_margin_pct  cogs_pct  
0                  7.06     70.73  
1 

In [9]:
yoy_query = """
CREATE TABLE financials_yoy AS
SELECT
    fiscal_year,
    net_sales,
    gross_margin_pct,
    operating_margin_pct,
    cogs_pct,
    sga_pct,

    -- YoY change in revenue (absolute, millions)
    net_sales - LAG(net_sales) OVER (ORDER BY period_ending)
        AS revenue_change_abs,

    -- YoY change in gross margin (percentage points)
    gross_margin_pct - LAG(gross_margin_pct) OVER (ORDER BY period_ending)
        AS gross_margin_change_pp,

    -- YoY change in operating margin (percentage points)
    operating_margin_pct - LAG(operating_margin_pct) OVER (ORDER BY period_ending)
        AS op_margin_change_pp,

    -- YoY change in COGS % (cost pressure indicator)
    cogs_pct - LAG(cogs_pct) OVER (ORDER BY period_ending)
        AS cogs_pct_change_pp
    
FROM financials_computed
order by period_ending;
""" 

conn.execute("DROP TABLE IF EXISTS financials_yoy")
conn.execute(yoy_query)
conn.commit()

yoy = pd.read_sql("SELECT * FROM financials_yoy", conn)
print(yoy)

  fiscal_year  net_sales  gross_margin_pct  operating_margin_pct  cogs_pct  \
0      FY2021      93561             29.27                  7.06     70.73   
1      FY2022     106005             29.28                  8.55     70.72   
2      FY2023     109120             24.57                  3.63     75.43   
3      FY2024     107412             27.54                  5.49     72.46   
4      FY2025     106566             28.21                  5.39     71.79   

   sga_pct  revenue_change_abs  gross_margin_change_pp  op_margin_change_pp  \
0    19.82                 NaN                     NaN                  NaN   
1    18.52             12444.0                    0.01                 1.49   
2    18.76              3115.0                   -4.71                -4.92   
3    19.80             -1708.0                    2.97                 1.86   
4    20.45              -846.0                    0.67                -0.10   

   cogs_pct_change_pp  
0                 NaN  
1       

In [11]:
decomp_query = """ 
SELECT
    fiscal_year,

    -- Revenue impact: if margins had stayed flat, how much would operating income
    -- have changed just from revenue movement?
    ROUND(
        (net_sales - LAG(net_sales) OVER (ORDER BY period_ending))
        * LAG(operating_margin_pct) OVER (ORDER BY period_ending)/100.00,
        0
    ) AS revenue_volumne_impact,

    -- Gross margin impact: how much did COGS changes 
    -- cost Target in dollar terms?
    ROUND(
        net_sales * (gross_margin_pct - LAG(gross_margin_pct) 
        OVER (ORDER BY period_ending)) / 100.0, 0 )
    AS gross_margin_impact,

    -- SGA impact: how much did overhead changes cost or save Target?
    ROUND(
        -net_sales * (sga_pct - LAG(sga_pct) OVER (ORDER BY period_ending)) / 
        100.0, 0
    ) AS sga_impact,

    -- Actual operating income change (should approximately = sum of above)
    operating_income - LAG(operating_income) OVER (ORDER BY period_ending)
        AS actual_oi_change

FROM financials_computed
ORDER BY period_ending;
"""

decomp = pd.read_sql(decomp_query, conn)
print(decomp)

  fiscal_year  revenue_volumne_impact  gross_margin_impact  sga_impact  \
0      FY2021                     NaN                  NaN         NaN   
1      FY2022                   879.0                 11.0      1378.0   
2      FY2023                   266.0              -5140.0      -262.0   
3      FY2024                   -62.0               3190.0     -1117.0   
4      FY2025                   -46.0                714.0      -693.0   

   actual_oi_change  
0               NaN  
1            2454.0  
2           -5102.0  
3            1940.0  
4            -161.0  


In [14]:
decomp_focus = decomp[decomp['fiscal_year'] == 'FY2023'].copy()

bridge_data = {
    'Category': [
        'FY2022 Operating Income',
        'Reenue Volume Effect',
        'Gross Margin Deterioation (COGS)',
        'SG&A Change',
        'FY2023 Operating Income'
    ],
    'Amount': [
        9062, # FY2022 baseline
        266, # revenue volume effect'
        -5140, # gross margin impact
        -262, # SGA impact
        3960 # FY2023 actual
    ],
    'Base':   [0, 9062, 9328, 4188, 0],
    'Bar':    [9062, 266, 5140, 262, 3960]
}

bridge_df = pd.DataFrame(bridge_data)
bridge_df.to_excel('target_variance_bridge.xlsx', index=False)
print("Bridge data exported.")

Bridge data exported.
